In [1]:
# ================================================================
#  MULTI-SIGNAL DROWSINESS DETECTION — COLAB TRAINING (FIXED)
#  Trains 2 CNN models: cnnEye.keras + cnnYawn.keras
#
#  IMPORTANT: Run ONLY Cell 1 first, then restart runtime,
#             then run all other cells one by one.
# ================================================================


# ════════════════════════════════════════════════════════════
#  CELL 1 — Setup (Run this FIRST, then restart runtime!)
# ════════════════════════════════════════════════════════════
!pip install -q opencv-python-headless Pillow
print("Done! Now click Runtime → Restart Runtime → then run Cell 2")


# ════════════════════════════════════════════════════════════
#  CELL 2 — All imports + verify
#  Run this after restarting runtime
# ════════════════════════════════════════════════════════════
import os, sys, shutil, glob, random, zipfile
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.image as mpimg

# Fix recursion limit
sys.setrecursionlimit(10000)

# Import tensorflow carefully
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'
import tensorflow as tf
from tensorflow.keras import layers, models, callbacks
from tensorflow.keras.models import load_model
from PIL import Image
from google.colab import files

print(f"✅ TensorFlow : {tf.__version__}")
print(f"✅ NumPy      : {np.__version__}")
print(f"✅ Python     : {sys.version}")
print(f"✅ All imports successful!")

# Settings
IMG_SIZE   = 24
BATCH_SIZE = 32
EPOCHS     = 20


Done! Now click Runtime → Restart Runtime → then run Cell 2
✅ TensorFlow : 2.19.0
✅ NumPy      : 2.0.2
✅ Python     : 3.12.13 (main, Mar  4 2026, 09:23:07) [GCC 11.4.0]
✅ All imports successful!


In [2]:
# ════════════════════════════════════════════════════════════
#  CELL 3 — Helper functions
# ════════════════════════════════════════════════════════════

def organize_dataset(src, dest, class_keywords):
    if os.path.exists(dest):
        shutil.rmtree(dest)
    for split in ['train', 'test']:
        for cls in set(class_keywords.values()):
            os.makedirs(f'{dest}/{split}/{cls}', exist_ok=True)

    all_imgs = (
        glob.glob(f'{src}/**/*.jpg',  recursive=True) +
        glob.glob(f'{src}/**/*.png',  recursive=True) +
        glob.glob(f'{src}/**/*.jpeg', recursive=True)
    )
    print(f"📸 Total images found: {len(all_imgs)}")

    buckets = {v: [] for v in set(class_keywords.values())}
    for p in all_imgs:
        parts = p.lower().replace('\\', '/').split('/')
        for kw, cls in class_keywords.items():
            if any(kw in part for part in parts):
                buckets[cls].append(p)
                break

    for cls, imgs in buckets.items():
        print(f"  {cls}: {len(imgs)} images")
        random.shuffle(imgs)
        split_idx = int(len(imgs) * 0.8)
        for i, p in enumerate(imgs[:split_idx]):
            ext = os.path.splitext(p)[1] or '.jpg'
            shutil.copy(p, f'{dest}/train/{cls}/{i:05d}{ext}')
        for i, p in enumerate(imgs[split_idx:]):
            ext = os.path.splitext(p)[1] or '.jpg'
            shutil.copy(p, f'{dest}/test/{cls}/{i:05d}{ext}')

    print(f"✅ Organized into {dest}/")
    for split in ['train','test']:
        for cls in set(class_keywords.values()):
            n = len(glob.glob(f'{dest}/{split}/{cls}/*'))
            print(f"   {split}/{cls}: {n}")


def load_images_numpy(dataset_path):
    classes = sorted([
        d for d in os.listdir(f'{dataset_path}/train')
        if os.path.isdir(f'{dataset_path}/train/{d}')
    ])
    print(f"  Classes: {classes}")

    def load_split(split):
        X, y = [], []
        for label, cls in enumerate(classes):
            paths = (
                glob.glob(f'{dataset_path}/{split}/{cls}/*.jpg')  +
                glob.glob(f'{dataset_path}/{split}/{cls}/*.png')  +
                glob.glob(f'{dataset_path}/{split}/{cls}/*.jpeg')
            )
            print(f"    {split}/{cls}: {len(paths)} images")
            for p in paths:
                try:
                    img = Image.open(p).convert('L').resize(
                        (IMG_SIZE, IMG_SIZE))
                    X.append(np.array(img, dtype=np.float32) / 255.0)
                    y.append(label)
                except:
                    pass
        X = np.array(X).reshape(-1, IMG_SIZE, IMG_SIZE, 1)
        y = tf.keras.utils.to_categorical(y, len(classes))
        return X, y

    print("  Loading train images...")
    X_train, y_train = load_split('train')
    print("  Loading test images...")
    X_test,  y_test  = load_split('test')
    print(f"  Train: {X_train.shape}  Test: {X_test.shape}")
    return X_train, y_train, X_test, y_test, classes


def build_cnn(n_classes):
    m = models.Sequential([
        layers.Conv2D(32,(3,3), activation='relu',
                      input_shape=(IMG_SIZE,IMG_SIZE,1)),
        layers.MaxPooling2D(2,2),
        layers.Dropout(0.25),
        layers.Conv2D(32,(3,3), activation='relu'),
        layers.MaxPooling2D(2,2),
        layers.Dropout(0.25),
        layers.Conv2D(64,(3,3), activation='relu', padding='same'),
        layers.Dropout(0.25),
        layers.Flatten(),
        layers.Dense(128, activation='relu'),
        layers.Dropout(0.5),
        layers.Dense(n_classes, activation='softmax')
    ])
    m.compile(optimizer='adam',
              loss='categorical_crossentropy',
              metrics=['accuracy'])
    return m


def train_model(dataset_path, model_name):
    print(f"\n{'='*50}")
    print(f"Training: {model_name}")
    print(f"{'='*50}")
    X_tr, y_tr, X_te, y_te, classes = load_images_numpy(dataset_path)
    model = build_cnn(len(classes))
    model.summary()
    os.makedirs('models', exist_ok=True)
    cbs = [
        callbacks.ModelCheckpoint(
            f'models/{model_name}',
            monitor='val_accuracy',
            save_best_only=True, verbose=1),
        callbacks.EarlyStopping(
            monitor='val_loss',
            patience=5,
            restore_best_weights=True, verbose=1)
    ]
    print(f"\n🚀 Training {model_name}...")
    history = model.fit(
        X_tr, y_tr,
        epochs=EPOCHS,
        batch_size=BATCH_SIZE,
        validation_split=0.2,
        callbacks=cbs
    )
    loss, acc = model.evaluate(X_te, y_te, verbose=0)
    print(f"\n✅ Accuracy: {acc*100:.2f}%")
    print(f"   Classes : {dict(enumerate(classes))}")
    return history, acc, classes


def preview(dataset_path, title):
    folders = sorted([f for f in glob.glob(f'{dataset_path}/train/*')
                      if os.path.isdir(f)])
    n = len(folders)
    fig, axes = plt.subplots(n, 5, figsize=(14, 3*n))
    fig.suptitle(title, fontsize=13)
    if n == 1: axes = [axes]
    for i, folder in enumerate(folders):
        imgs = glob.glob(f'{folder}/*')
        samples = random.sample(imgs, min(5, len(imgs)))
        for j, p in enumerate(samples):
            try:
                axes[i][j].imshow(mpimg.imread(p), cmap='gray')
                axes[i][j].set_title(os.path.basename(folder), fontsize=8)
            except: pass
            axes[i][j].axis('off')
    plt.tight_layout()
    plt.savefig('/tmp/preview.png', dpi=80, bbox_inches='tight')
    plt.show()

print("✅ All helper functions ready!")

✅ All helper functions ready!


In [3]:
# ════════════════════════════════════════════════════════════
#  CELL 4 — Upload Eyes Dataset ZIP
#  Download from:
#  kaggle.com/datasets/kutaykutlu/drowsiness-detection
# ════════════════════════════════════════════════════════════

print("📁 Upload your EYES dataset ZIP...")
uploaded = files.upload()

for fname in uploaded.keys():
    if os.path.exists('raw_eyes'): shutil.rmtree('raw_eyes')
    print(f"📦 Extracting {fname}...")
    with zipfile.ZipFile(fname, 'r') as z:
        z.extractall('raw_eyes')
    print("✅ Done!")

print("\n📂 Structure:")
for root, dirs, fs in os.walk('raw_eyes'):
    level = root.replace('raw_eyes','').count(os.sep)
    if level < 4:
        print('  '*level + os.path.basename(root) + f'/ ({len(fs)} files)')


# ════════════════════════════════════════════════════════════
#  CELL 5 — Organize Eyes Dataset
# ════════════════════════════════════════════════════════════

organize_dataset(
    src='raw_eyes',
    dest='dataset_eyes',
    class_keywords={
        'open':   'open_eyes',
        'closed': 'closed_eyes',
    }
)
preview('dataset_eyes', '👁️ Eyes — open_eyes vs closed_eyes')

📁 Upload your EYES dataset ZIP...


Saving archive (1).zip to archive (1).zip
📦 Extracting archive (1).zip...
✅ Done!

📂 Structure:
raw_eyes/ (0 files)
  closed_eye/ (24000 files)
  open_eye/ (24000 files)
📸 Total images found: 48000
  closed_eyes: 24000 images
  open_eyes: 24000 images
✅ Organized into dataset_eyes/
   train/closed_eyes: 19200
   train/open_eyes: 19200
   test/closed_eyes: 4800
   test/open_eyes: 4800


/tmp/ipykernel_275/1678212793.py:151: UserWarning: Glyph 128065 (\N{EYE}) missing from font(s) DejaVu Sans.
  plt.tight_layout()
/tmp/ipykernel_275/1678212793.py:152: UserWarning: Glyph 128065 (\N{EYE}) missing from font(s) DejaVu Sans.
  plt.savefig('/tmp/preview.png', dpi=80, bbox_inches='tight')


In [4]:
# ════════════════════════════════════════════════════════════
#  CELL 6 — Upload Yawn Dataset ZIP
#  Download from:
#  kaggle.com/datasets/deepankarvarma/yawning-dataset-classification
# ════════════════════════════════════════════════════════════

print("📁 Upload your YAWN dataset ZIP...")
uploaded = files.upload()

for fname in uploaded.keys():
    if os.path.exists('raw_yawn'): shutil.rmtree('raw_yawn')
    print(f"📦 Extracting {fname}...")
    with zipfile.ZipFile(fname, 'r') as z:
        z.extractall('raw_yawn')
    print("✅ Done!")

print("\n📂 Structure:")
for root, dirs, fs in os.walk('raw_yawn'):
    level = root.replace('raw_yawn','').count(os.sep)
    if level < 4:
        print('  '*level + os.path.basename(root) + f'/ ({len(fs)} files)')

📁 Upload your YAWN dataset ZIP...


Saving archive (2).zip to archive (2).zip
📦 Extracting archive (2).zip...
✅ Done!

📂 Structure:
raw_yawn/ (0 files)
  no yawn/ (2591 files)
  yawn/ (2528 files)


In [9]:
import os, glob

all_imgs = (glob.glob('raw_yawn/**/*.jpg', recursive=True) +
            glob.glob('raw_yawn/**/*.png', recursive=True))

# Show all unique parent folder names
folder_names = set()
for p in all_imgs:
    parts = p.replace('\\','/').split('/')
    folder_names.add(parts[-2])

print(f"📂 Exact folder names: {folder_names}")

📂 Exact folder names: {'yawn', 'no yawn'}


In [11]:
import os, shutil, glob, random

# Clean previous
if os.path.exists('dataset_yawn'):
    shutil.rmtree('dataset_yawn')

# Create folders
for split in ['train', 'test']:
    for cls in ['yawn', 'no_yawn']:
        os.makedirs(f'dataset_yawn/{split}/{cls}', exist_ok=True)

# Find all images directly by folder name
yawn_imgs    = []
no_yawn_imgs = []

for p in glob.glob('raw_yawn/**/*.*', recursive=True):
    parent = os.path.basename(os.path.dirname(p))
    if parent == 'yawn':
        yawn_imgs.append(p)
    elif parent == 'no yawn':      # exact match with space
        no_yawn_imgs.append(p)

print(f"✅ yawn    : {len(yawn_imgs)} images")
print(f"✅ no_yawn : {len(no_yawn_imgs)} images")

# Split 80/20 and copy
def copy_split(imgs, cls):
    random.shuffle(imgs)
    split = int(len(imgs) * 0.8)
    for i, p in enumerate(imgs[:split]):
        ext = os.path.splitext(p)[1] or '.jpg'
        shutil.copy(p, f'dataset_yawn/train/{cls}/{i:05d}{ext}')
    for i, p in enumerate(imgs[split:]):
        ext = os.path.splitext(p)[1] or '.jpg'
        shutil.copy(p, f'dataset_yawn/test/{cls}/{i:05d}{ext}')
    print(f"   train/{cls}: {split}  test/{cls}: {len(imgs)-split}")

copy_split(yawn_imgs,    'yawn')
copy_split(no_yawn_imgs, 'no_yawn')
print("\n✅ Yawn dataset organized!")

✅ yawn    : 2528 images
✅ no_yawn : 2591 images
   train/yawn: 2022  test/yawn: 506
   train/no_yawn: 2072  test/no_yawn: 519

✅ Yawn dataset organized!


In [13]:
# ════════════════════════════════════════════════════════════
#  CELL 7 — Organize Yawn Dataset
# ════════════════════════════════════════════════════════════

# ── Cell 7 — Organize Yawn Dataset (FIXED) ────────────────
import os, shutil, glob, random

# Clean previous
if os.path.exists('dataset_yawn'):
    shutil.rmtree('dataset_yawn')

# Create folders
for split in ['train', 'test']:
    for cls in ['yawn', 'no_yawn']:
        os.makedirs(f'dataset_yawn/{split}/{cls}', exist_ok=True)

# Find images using EXACT parent folder name match
yawn_imgs    = []
no_yawn_imgs = []

for p in glob.glob('raw_yawn/**/*.*', recursive=True):
    parent = os.path.basename(os.path.dirname(p))
    if parent == 'yawn':
        yawn_imgs.append(p)
    elif parent == 'no yawn':      # exact match — space not underscore
        no_yawn_imgs.append(p)

print(f"✅ yawn    : {len(yawn_imgs)} images")
print(f"✅ no_yawn : {len(no_yawn_imgs)} images")

# 80% train / 20% test split
def copy_split(imgs, cls):
    random.shuffle(imgs)
    split = int(len(imgs) * 0.8)
    for i, p in enumerate(imgs[:split]):
        ext = os.path.splitext(p)[1] or '.jpg'
        shutil.copy(p, f'dataset_yawn/train/{cls}/{i:05d}{ext}')
    for i, p in enumerate(imgs[split:]):
        ext = os.path.splitext(p)[1] or '.jpg'
        shutil.copy(p, f'dataset_yawn/test/{cls}/{i:05d}{ext}')
    print(f"   train/{cls}: {split}  test/{cls}: {len(imgs)-split}")

copy_split(yawn_imgs,    'yawn')
copy_split(no_yawn_imgs, 'no_yawn')
print("\n✅ Yawn dataset ready.")
preview('dataset_yawn', '😮 Yawn — yawn vs no_yawn')

✅ yawn    : 2528 images
✅ no_yawn : 2591 images
   train/yawn: 2022  test/yawn: 506
   train/no_yawn: 2072  test/no_yawn: 519

✅ Yawn dataset ready.


In [14]:
# ════════════════════════════════════════════════════════════
#  CELL 8 — Train Eye Model
# ════════════════════════════════════════════════════════════

history_eye, acc_eye, cls_eye = train_model(
    'dataset_eyes', 'cnnEye.keras')

print(f"\n👁️  Eye classes : {dict(enumerate(cls_eye))}")


Training: cnnEye.keras
  Classes: ['closed_eyes', 'open_eyes']
  Loading train images...
    train/closed_eyes: 19200 images
    train/open_eyes: 19200 images
  Loading test images...
    test/closed_eyes: 4800 images
    test/open_eyes: 4800 images
  Train: (38400, 24, 24, 1)  Test: (9600, 24, 24, 1)


Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d_3 (Conv2D)               │ (None, 22, 22, 32)     │           320 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_2 (MaxPooling2D)  │ (None, 11, 11, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_4 (Dropout)             │ (None, 11, 11, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_4 (Conv2D)               │ (None, 9, 9, 32)       │         9,248 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_3 (MaxPooling2D)  │ (None, 4, 4, 32)       │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_5 (Dropout)             │ (None, 4, 4, 32)       │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_5 (Conv2D)               │ (None, 4, 4, 64)       │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_6 (Dropout)             │ (None, 4, 4, 64)       │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten_1 (Flatten)             │ (None, 1024)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 128)            │       131,200 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_7 (Dropout)             │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 2)              │           258 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 159,522 (623.13 KB)

 Trainable params: 159,522 (623.13 KB)

 Non-trainable params: 0 (0.00 B)


🚀 Training cnnEye.keras...
Epoch 1/20
959/960 ━━━━━━━━━━━━━━━━━━━━ 0s 24ms/step - accuracy: 0.8387 - loss: 0.3519
Epoch 1: val_accuracy improved from None to 0.96172, saving model to models/cnnEye.keras

Epoch 1: finished saving model to models/cnnEye.keras
960/960 ━━━━━━━━━━━━━━━━━━━━ 26s 25ms/step - accuracy: 0.9067 - loss: 0.2272 - val_accuracy: 0.9617 - val_loss: 0.1148
Epoch 2/20
958/960 ━━━━━━━━━━━━━━━━━━━━ 0s 23ms/step - accuracy: 0.9547 - loss: 0.1236
Epoch 2: val_accuracy did not improve from 0.96172
960/960 ━━━━━━━━━━━━━━━━━━━━ 24s 25ms/step - accuracy: 0.9598 - loss: 0.1108 - val_accuracy: 0.9585 - val_loss: 0.1181
Epoch 3/20
958/960 ━━━━━━━━━━━━━━━━━━━━ 0s 23ms/step - accuracy: 0.9653 - loss: 0.0904
Epoch 3: val_accuracy improved from 0.96172 to 0.97474, saving model to models/cnnEye.keras

Epoch 3: finished saving model to models/cnnEye.keras
960/960 ━━━━━━━━━━━━━━━━━━━━ 25s 26ms/step - accuracy: 0.9673 - loss: 0.0862 - val_accuracy: 0.9747 - val_loss: 0.0665
Epoch 4/20
9

In [15]:
# ════════════════════════════════════════════════════════════
#  CELL 9 — Train Yawn Model
# ════════════════════════════════════════════════════════════

history_yawn, acc_yawn, cls_yawn = train_model(
    'dataset_yawn', 'cnnYawn.keras')

print(f"\n😮  Yawn classes: {dict(enumerate(cls_yawn))}")


Training: cnnYawn.keras
  Classes: ['no_yawn', 'yawn']
  Loading train images...
    train/no_yawn: 2072 images
    train/yawn: 2022 images
  Loading test images...
    test/no_yawn: 519 images
    test/yawn: 506 images
  Train: (4094, 24, 24, 1)  Test: (1025, 24, 24, 1)


Model: "sequential_2"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d_6 (Conv2D)               │ (None, 22, 22, 32)     │           320 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_4 (MaxPooling2D)  │ (None, 11, 11, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_8 (Dropout)             │ (None, 11, 11, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_7 (Conv2D)               │ (None, 9, 9, 32)       │         9,248 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_5 (MaxPooling2D)  │ (None, 4, 4, 32)       │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_9 (Dropout)             │ (None, 4, 4, 32)       │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_8 (Conv2D)               │ (None, 4, 4, 64)       │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_10 (Dropout)            │ (None, 4, 4, 64)       │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten_2 (Flatten)             │ (None, 1024)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_4 (Dense)                 │ (None, 128)            │       131,200 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_11 (Dropout)            │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_5 (Dense)                 │ (None, 2)              │           258 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 159,522 (623.13 KB)

 Trainable params: 159,522 (623.13 KB)

 Non-trainable params: 0 (0.00 B)


🚀 Training cnnYawn.keras...
Epoch 1/20
102/103 ━━━━━━━━━━━━━━━━━━━━ 0s 28ms/step - accuracy: 0.6681 - loss: 0.6043
Epoch 1: val_accuracy improved from None to 0.83883, saving model to models/cnnYawn.keras

Epoch 1: finished saving model to models/cnnYawn.keras
103/103 ━━━━━━━━━━━━━━━━━━━━ 6s 32ms/step - accuracy: 0.7603 - loss: 0.5099 - val_accuracy: 0.8388 - val_loss: 0.2994
Epoch 2/20
103/103 ━━━━━━━━━━━━━━━━━━━━ 0s 21ms/step - accuracy: 0.8778 - loss: 0.3142
Epoch 2: val_accuracy did not improve from 0.83883
103/103 ━━━━━━━━━━━━━━━━━━━━ 4s 23ms/step - accuracy: 0.8919 - loss: 0.2910 - val_accuracy: 0.7338 - val_loss: 0.5219
Epoch 3/20
101/103 ━━━━━━━━━━━━━━━━━━━━ 0s 32ms/step - accuracy: 0.9303 - loss: 0.2136
Epoch 3: val_accuracy did not improve from 0.83883
103/103 ━━━━━━━━━━━━━━━━━━━━ 3s 33ms/step - accuracy: 0.9316 - loss: 0.2077 - val_accuracy: 0.7900 - val_loss: 0.4930
Epoch 4/20
102/103 ━━━━━━━━━━━━━━━━━━━━ 0s 32ms/step - accuracy: 0.9401 - loss: 0.1713
Epoch 4: val_accuracy

In [16]:
# ════════════════════════════════════════════════════════════
#  CELL 10 — Plot Accuracy Graphs
# ════════════════════════════════════════════════════════════

fig, axes = plt.subplots(2, 2, figsize=(14, 10))

for row, (history, title) in enumerate([
    (history_eye,  '👁️  Eye Model'),
    (history_yawn, '😮  Yawn Model'),
]):
    axes[row][0].plot(history.history['accuracy'],
                      label='Train', color='blue')
    axes[row][0].plot(history.history['val_accuracy'],
                      label='Val',   color='orange')
    axes[row][0].set_title(f'{title} Accuracy')
    axes[row][0].legend(); axes[row][0].grid(True)

    axes[row][1].plot(history.history['loss'],
                      label='Train', color='blue')
    axes[row][1].plot(history.history['val_loss'],
                      label='Val',   color='orange')
    axes[row][1].set_title(f'{title} Loss')
    axes[row][1].legend(); axes[row][1].grid(True)

plt.tight_layout()
plt.show()

print(f"\n📊 Results:")
print(f"   👁️  Eye  Model: {acc_eye*100:.2f}%")
print(f"   😮  Yawn Model: {acc_yawn*100:.2f}%")

/tmp/ipykernel_275/4054710258.py:25: UserWarning: Glyph 128065 (\N{EYE}) missing from font(s) DejaVu Sans.
  plt.tight_layout()



📊 Results:
   👁️  Eye  Model: 99.19%
   😮  Yawn Model: 95.41%


/usr/local/lib/python3.12/dist-packages/IPython/core/events.py:89: UserWarning: Glyph 128065 (\N{EYE}) missing from font(s) DejaVu Sans.
  func(*args, **kwargs)


In [17]:
















# ════════════════════════════════════════════════════════════
#  CELL 11 — Download Both Models
# ════════════════════════════════════════════════════════════

files.download('models/cnnEye.keras')
files.download('models/cnnYawn.keras')

print("\n✅ Both models downloaded!")
print("   Put cnnEye.keras + cnnYawn.keras in same folder as detect_advanced.py")
print("   Then run: python detect_advanced.py")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>


✅ Both models downloaded!
   Put cnnEye.keras + cnnYawn.keras in same folder as detect_advanced.py
   Then run: python detect_advanced.py
